## <b><font color='darkblue'>Preface</font></b>
([source](https://adk.dev/skills/)) <font size='3ptx'><b>An agent Skill is a self-contained unit of functionality that an ADK agent can use to perform a specific task.</b> An agent Skill encapsulates the necessary instructions, resources, and tools required for a task, based on the Agent Skill specification.</font>

<b>The structure of a Skill allows it to be loaded incrementally to minimize the impact on the operating context window of the agent.</b>

In [2]:
import agent_utils
import os
import pathlib
from pprint import pprint
import warnings

from google.adk import Agent
from google.adk.skills import load_skill_from_dir
from google.adk.tools import skill_toolset
from google.adk.tools.skill_toolset import SkillToolset

AgentBroker = agent_utils.AgentBroker
CURRENT_WORK_DIR: str = os.getcwd()
print(CURRENT_WORK_DIR)

# Suppress the specific Google API key precedence warning
warnings.filterwarnings(
    "ignore", 
    message=".*Both GOOGLE_API_KEY and GEMINI_API_KEY are set.*"
)

/usr/local/google/home/johnkclee/Github/ml_articles/google/agent_development_kit


## <b><font color='darkblue'>Get started</font></b>
<font size='3ptx'><b>Use the <font color='blue'>SkillToolset</font> class to make one or more Skills available to your agent</b>. You can define [**skills in code**](https://adk.dev/skills/#inline-skills) or [**load skills from a filesystem**](https://adk.dev/skills/#filesystem-skills).</font>

### <b><font color='darkgreen'>Function `load_skill_from_dir`</font></b>
<font size='3ptx'><b>This function is a utility used to load a complete Agent Skill from a local directory</b>. It follows a specific directory structure and convention to assemble the skill's metadata, instructions, and resources.</font>

#### <b>Expected Directory Structure</b>
Example Directory Structure:
```
my-cool-skill/
├── SKILL.md          # Frontmatter + Instructions
├── references/       # Optional: extra docs
│   └── workflow.md
├── assets/           # Optional: static files
│   └── schema.json
└── scripts/          # Optional: executable code
    └── run.sh
```

#### <b>Parsing Flow and Class Diagram<>
The function is defined in [**google/adk/skills/**](https://adk.dev/api-reference/python/google-adk.html#module-google.adk.utils) and performs the following steps:
1. **Path Resolution**: It takes a directory path (as a string or `pathlib.Path`) and resolves it to an absolute path.
2. **SKILL.md Parsing**: It looks for a <font color='olive'>SKILL.md</font> (or skill.md) file in the root of the directory. This file must start with a YAML frontmatter block (<font color='brown'>delimited by `---`</font>).
    - The Frontmatter contains metadata like `name`, `description`, and `allowed-tools`.
    - The Body of the markdown file becomes the skill's primary `instructions`.
3. **Validation**: It validates that the name field in the frontmatter matches the name of the directory.
4. **Resource Loading**: It recursively scans three specific subdirectories if they exist.
    - **`references/`**: Additional markdown guidance or workflows.
    - **`assets/`**: Static materials like schemas or templates.
    - **`scripts/`**: Executable scripts (wrapped in <b><font color='blue'>models.Script</font></b> objects).
5. **Object Construction**: It returns a fully populated <b><font color='blue'>models.Skill</font></b> object.
```python
class Skill(BaseModel):
  """Complete skill representation including frontmatter, instructions, and resources.

  A skill combines:
  - L1: Frontmatter for discovery (name, description).
  - L2: Instructions from SKILL.md body, loaded when skill is triggered.
  - L3: Resources including additional instructions, assets, and scripts,
  loaded as needed.

  Attributes:
      frontmatter: Parsed skill frontmatter from SKILL.md.
      instructions: L2 skill content: markdown instruction from SKILL.md body.
      resources: L3 skill content: additional instructions, assets, and scripts.
  """
    
  frontmatter: Frontmatter
  instructions: str
  resources: Resources = Resources()
```

```mermaid
classDiagram
    class Skill {
        +Frontmatter frontmatter
        +str instructions
        +Resources resources
        +name() str
        +description() str
    }

    class Frontmatter {
        +str name
        +str description
        +str? license
        +str? compatibility
        +str? allowed_tools
        +dict metadata
    }

    class Resources {
        +dict[str, str|bytes] references
        +dict[str, str|bytes] assets
        +dict[str, Script] scripts
        +get_reference(id)
        +get_asset(id)
        +get_script(id)
    }

    class Script {
        +str src
        +__str__() str
    }

    Skill *-- Frontmatter : contains
    Skill *-- Resources : contains
    Resources *-- Script : contains

```

### <b><font color='darkgreen'>Usage Example</font></b>

Below codesnippet demonstrate on to load in skills from file system:

In [2]:
weather_skill = load_skill_from_dir(
    pathlib.Path(CURRENT_WORK_DIR) / "skills" / "weather-skill"
)

In [7]:
weather_skill.__class__

google.adk.skills.models.Skill

In [9]:
pprint(weather_skill.model_dump())

{'frontmatter': {'allowed_tools': None,
                 'compatibility': None,
                 'description': 'A skill that provides weather information '
                                'based on reference data.',
                 'license': None,
                 'metadata': {'adk_additional_tools': ['get_wind_speed']},
                 'name': 'weather-skill'},
 'instructions': "Step 1: Check 'references/weather_info.md' for the current "
                 'weather.\n'
                 'Step 2: If humidity is requested, use run '
                 "'scripts/get_humidity.py' with the `location` argument.\n"
                 'Step 3: If wind speed is requested, use the `get_wind_speed` '
                 'tool.\n'
                 'Step 4: Provide the update to the user.',
 'resources': {'assets': {},
               'references': {'weather_info.md': '# Weather Information\n'
                                                 '\n'
                                                 '- **Loc

## <b><font color='darkblue'>Skill structure</font></b>
<font size='3ptx'><b>([source](https://adk.dev/skills/#skill-structure)) The Skills feature allows you to create modular packages of Skill instructions and resources that agents can load on demand.</b> This approach helps you organize your agent's capabilities and optimize the context window by only loading instructions when they are needed. The structure of Skills is organized into three levels:</font>

### <font color='darkgreen'><b>Key Components Breakdown</b></font>
<b><font color='blue'>models.Skill</font></b>: The root container. It represents the "Complete Skill" (L1 + L2 + L3).
- <b>L1 (Metadata)</b>: Provides metadata for skill discovery. This information is defined in the frontmatter section of the <font color='olive'>SKILL.md</font> file and includes properties such as the Skill name and description.
- <b>L2 (Instructions)</b>: Contains the primary instructions for the Skill, loaded when the Skill is triggered by the agent. This information is defined in the body of the <font color='olive'>SKILL.md</font> file.
- <b>L3 (Resources)</b>: Includes additional resources such as reference materials, assets, and scripts that can be loaded as needed. These resources are organized into the following directories:
    - <b>`references/`</b>: Additional Markdown files with extended instructions, workflows, or guidance.
    - <b>`assets/`</b>: Resource materials such as database schemas, API documentation, templates, or examples.
    - <b>`scripts/`</b>: Executable scripts supported by the agent runtime.

### <b><font color='darkgreen'>System instructions for using skills</font></b>
<font size='3ptx'>The [**SkillToolset**](https://adk.dev/api-reference/python/google-adk.html#google.adk.tools.skill_toolset.SkillToolset) provides a default system instruction to the agent that outlines how it should interact with skills.</font>

These instructions include the following key points:
- You must use the `load_skill` tool to read a skill's instructions before using it.
- You must follow the instructions in the skill definition exactly.
- You must use the `load_skill_resource` tool to view files within a skill's directory.
- You must use the `run_skill_script` to run scripts from a skill's <font color='olive'>scripts/</font> directory.

### <b><font color='darkgreen'>Skill validation</font></b>
The frontmatter of a skill's <font color='olive'>SKILL.md</font> file is validated to ensure that it meets the following requirements:

- **name**:
    - Must be 64 characters or less.
    - Must be in lowercase, kebab-case (a-z, 0-9, and hyphens).
    - Must not have leading, trailing, or consecutive hyphens.
- **description**:
    - Must not be empty.
    - Must be 1024 characters or less.

### <b><font color='darkgreen'>Skills directory structure</font></b>
<font size='3ptx'>The following directory structure shows the recommended way to include Skills in your ADK agent project.</font>

The <font color='olive'>example-skill/</font> directory shown below, and any parallel Skill directories, must follow the Agent Skill specification file structure. Only the <font color='olive'>SKILL.md</font> file is required:
```text
my_agent/
    agent.py (or agent.ts / main.go)
    .env
    skills/
        example-skill/        # Skill
            SKILL.md          # main instructions (required)
            references/
                REFERENCE.md  # detailed API reference
                FORMS.md      # form-filling guide
                *.md          # domain-specific information
            assets/
                *.*           # templates, images, data
            scripts/
                *.py          # utility scripts (Python)
                *.js          # utility scripts (JavaScript)
                *.ts          # utility scripts (TypeScript)
```

## <b><font color='darkblue'>Skill sources</font></b>
<font size='3ptx'>([source](https://adk.dev/skills/#skill-sources)) You can define [skills within the code](https://adk.dev/skills/#inline-skills) or read [skills from a filesystem](https://adk.dev/skills/#filesystem-skills).</font>

### <b><font color='darkgreen'>Define Skills in code</font></b>
<font size='3ptx'>([source](https://adk.dev/skills/#inline-skills)) You can define Skills within the code of your agent, as shown below:</font>

In [3]:
from google.adk.skills import models

greeting_skill = models.Skill(
    frontmatter=models.Frontmatter(
        name="greeting-skill",
        description=(
            "A friendly greeting skill that can say hello to a specific person."
        ),
    ),
    instructions=(
        "Step 1: Read the 'references/hello_world.txt' file to understand how"
        " to greet the user. Step 2: Return a greeting based on the reference."
    ),
    resources=models.Resources(
        references={
            "hello_world.txt": "Hello! So glad to have you here!",
            "example.md": "This is an example reference.",
        },
    ),
)

In [4]:
# Package your skills into a toolset bundle
my_skill_toolset = SkillToolset(skills=[greeting_skill])

# Create the root agent and provision it with the skill toolset
root_agent = Agent(
    model="gemini-3.5-flash",
    name="support_agent",
    description="An AI assistant equipped with special text processing skills.",
    instruction="You are a corporate support desk bot. Use your toolsets when processing logs.",
    tools=[my_skill_toolset]
)

In [5]:
# Create agent broker
broker = AgentBroker(root_agent)

In [6]:
resp = await broker.interact("Hello!")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
print(resp.text)

Hello! So glad to have you here! How can I assist you today?


In [9]:
print(resp.thoughts)

In [18]:
# print(resp.events)

### <b><font color='darkgreen'>Read Skills from filesystem</font></b>

In [15]:
# Loaded skills:
greeting_skill = load_skill_from_dir(
    pathlib.Path(CURRENT_WORK_DIR) / "skills" / "greeting-skill"
)
weather_skill = load_skill_from_dir(
    pathlib.Path(CURRENT_WORK_DIR) / "skills" / "weather-skill"
)

my_skill_toolset = skill_toolset.SkillToolset(
    skills=[weather_skill, greeting_skill],
)

In [14]:
# Create agent broker:
root_agent = Agent(
    model="gemini-3.5-flash",
    name="support_agent",
    description="An AI assistant equipped with special text processing skills.",
    instruction="You are a corporate support desk bot. Use your toolsets when processing logs.",
    tools=[my_skill_toolset]
)
broker = AgentBroker(root_agent)

In [16]:
resp = await broker.interact("What is the humidity of Taipei!")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [17]:
print(resp.text)

Based on the weather information for **Taipei, TW**:

*   **Condition:** Sunny ☀️
*   **Temperature:** 72°F (22°C)
*   **Forecast:** Clear skies all day.

To retrieve the humidity, the system tried to execute the simulation script (`scripts/get_humidity.py`). However, because no code executor is currently configured in this environment to run scripts, the script could not be executed directly. 

Based on the contents of the simulation script, the simulated humidity for **Taipei** is **45%**.


## <b><font color='darkblue'>Next steps</font></b>
<font size='3ptx'>Check out these resources for building agents with Skills:</font>

- [Skills in Python - code sample](https://github.com/google/adk-python/tree/main/contributing/samples/environment_and_skills/skills_agent)
- [Skills in Go - code sample](https://github.com/google/adk-go/tree/main/examples/skills)
- [Agent Skills specification documentation](https://agentskills.io/)

## <b><font color='darkblue'>Supplement</font></b>
- [**G4D - Developer’s Guide to Building ADK Agents with Skills**](https://developers.googleblog.com/developers-guide-to-building-adk-agents-with-skills/)
> Your AI agent can follow instructions. But can it write new ones? [**Agent Development Kit**](https://adk.dev/) (ADK)'s [**SkillToolset**](https://google.github.io/adk-docs/skills/) enables agents to load domain expertise on demand...